In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, TransformerConv
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter


In [2]:
# --- 1. 超参数配置 ---
hparams = {
    'dataset': 'zoo',
    'threshold_pos': 4000,
    'threshold_neg': 49000,
    'gcn_hidden_channels': 16,
    'transformer_hidden_channels': 16,
    'heads': 4,
    'cpe_profile_bins': 8,
    'learning_rate': 0.005,
    'weight_decay': 5e-4,
    'epochs': 150,
    'dropout': 0.5,
    'seed': 42
}

# car 常用阈值: threshold_pos=500, threshold_neg=20000
# zoo 常用阈值: threshold_pos=4000, threshold_neg=49000

torch.manual_seed(hparams['seed'])
np.random.seed(hparams['seed'])


In [3]:
# --- 2. TensorBoard 设置 ---
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
log_dir_name = f"../runs/{hparams['dataset']}_gcn_transformer_cpe_profile{hparams['cpe_profile_bins']}_{timestamp}"
writer = SummaryWriter(log_dir_name)
print(f"TensorBoard 日志将保存在: {log_dir_name}")


TensorBoard 日志将保存在: ../runs/zoo_gcn_transformer_cpe_profile8_20260615-163536


In [4]:
# --- 3. 标签、CPE 与边权处理函数 ---
def load_labels(base_path, dataset_name, expected_num_nodes):
    candidate_paths = [
        f"{base_path}{dataset_name}.data",
        f"{base_path}{dataset_name}.data.csv",
    ]

    for labels_path in candidate_paths:
        try:
            df = pd.read_csv(labels_path, header=None)
        except FileNotFoundError:
            continue

        df = df.dropna(axis=1, how='all')
        if len(df) != expected_num_nodes:
            continue
        return df.iloc[:, -1].values

    raise ValueError(f"无法读取与对象数量 {expected_num_nodes} 匹配的标签文件: {candidate_paths}")


def normalize_edge_attr(edge_attr):
    edge_attr = torch.log1p(edge_attr.float())
    if edge_attr.numel() == 0:
        return edge_attr

    min_value = edge_attr.min()
    max_value = edge_attr.max()
    if max_value > min_value:
        edge_attr = (edge_attr - min_value) / (max_value - min_value)
    return edge_attr


def load_depth_profile_cpe(base_path, dataset_name, profile_bins):
    pos_cpe_path = f"{base_path}{dataset_name}_CPE_A_plus_depth_profile{profile_bins}.csv"
    neg_cpe_path = f"{base_path}{dataset_name}_CPE_A_negative_depth_profile{profile_bins}.csv"

    cpe_pos_numpy = np.loadtxt(pos_cpe_path, delimiter=',')
    cpe_neg_numpy = np.loadtxt(neg_cpe_path, delimiter=',')

    if cpe_pos_numpy.ndim == 1:
        cpe_pos_numpy = cpe_pos_numpy.reshape(1, -1)
    if cpe_neg_numpy.ndim == 1:
        cpe_neg_numpy = cpe_neg_numpy.reshape(1, -1)

    cpe_pos = torch.tensor(cpe_pos_numpy, dtype=torch.float)
    cpe_neg = torch.tensor(cpe_neg_numpy, dtype=torch.float)
    return cpe_pos, cpe_neg


In [5]:
# --- 4. 数据加载与预处理函数 (原始特征 + profile8 CPE + 正负概念图) ---
def load_and_prepare_data(dataset_name, threshold_pos, threshold_neg, cpe_profile_bins):
    base_path = f'../data/{dataset_name}/'

    features_path = f"{base_path}{dataset_name}.data.cleaned.csv"
    x_numpy = np.loadtxt(features_path, delimiter=',')
    x_features = torch.tensor(x_numpy, dtype=torch.float)
    num_nodes = x_features.shape[0]

    adj_matrix_pos_path = f"{base_path}{dataset_name}_A_plus_UG.csv"
    a_plus_pos_numpy = np.loadtxt(adj_matrix_pos_path, delimiter=',')
    if a_plus_pos_numpy.shape != (num_nodes, num_nodes):
        raise ValueError(f"正概念邻接矩阵尺寸应为 {(num_nodes, num_nodes)}，实际为 {a_plus_pos_numpy.shape}")
    a_plus_pos = torch.tensor(a_plus_pos_numpy, dtype=torch.float)
    a_plus_pos[a_plus_pos <= threshold_pos] = 0
    a_plus_pos.fill_diagonal_(0)
    edge_index_pos, edge_weight_pos = dense_to_sparse(a_plus_pos)
    edge_weight_pos = normalize_edge_attr(edge_weight_pos)

    adj_matrix_neg_path = f"{base_path}{dataset_name}_A_negative_UG.csv"
    a_plus_neg_numpy = np.loadtxt(adj_matrix_neg_path, delimiter=',')
    if a_plus_neg_numpy.shape != (num_nodes, num_nodes):
        raise ValueError(f"负概念邻接矩阵尺寸应为 {(num_nodes, num_nodes)}，实际为 {a_plus_neg_numpy.shape}")
    a_plus_neg = torch.tensor(a_plus_neg_numpy, dtype=torch.float)
    a_plus_neg[a_plus_neg <= threshold_neg] = 0
    a_plus_neg.fill_diagonal_(0)
    edge_index_neg, edge_weight_neg = dense_to_sparse(a_plus_neg)
    edge_weight_neg = normalize_edge_attr(edge_weight_neg)

    cpe_pos, cpe_neg = load_depth_profile_cpe(base_path, dataset_name, cpe_profile_bins)
    if cpe_pos.shape[0] != num_nodes or cpe_neg.shape[0] != num_nodes:
        raise ValueError(
            f"CPE 行数必须和对象数量一致: num_nodes={num_nodes}, "
            f"cpe_pos={cpe_pos.shape[0]}, cpe_neg={cpe_neg.shape[0]}"
        )

    print(f"原始特征维度: {x_features.shape[1]}")
    print(f"正概念 CPE 维度: {cpe_pos.shape[1]}")
    print(f"负概念 CPE 维度: {cpe_neg.shape[1]}")
    print(f"正概念图边数: {edge_index_pos.shape[1]}")
    print(f"负概念图边数: {edge_index_neg.shape[1]}")

    labels_numpy = load_labels(base_path, dataset_name, num_nodes)

    encoder = LabelEncoder()
    y_numpy = encoder.fit_transform(labels_numpy)
    y = torch.tensor(y_numpy, dtype=torch.long)
    if num_nodes != len(y):
        raise ValueError(f"标签数量必须和对象数量一致: num_nodes={num_nodes}, labels={len(y)}")

    data = Data(
        x_raw=x_features,
        cpe_pos=cpe_pos,
        cpe_neg=cpe_neg,
        y=y,
        edge_index_pos=edge_index_pos,
        edge_weight_pos=edge_weight_pos,
        edge_attr_pos=edge_weight_pos.view(-1, 1),
        edge_index_neg=edge_index_neg,
        edge_weight_neg=edge_weight_neg,
        edge_attr_neg=edge_weight_neg.view(-1, 1),
        num_nodes=num_nodes
    )

    num_train = int(num_nodes * 0.6)
    num_val = int(num_nodes * 0.2)
    indices = torch.randperm(num_nodes)
    data.train_mask = torch.zeros(num_nodes, dtype=torch.bool); data.train_mask[indices[:num_train]] = True
    data.val_mask = torch.zeros(num_nodes, dtype=torch.bool); data.val_mask[indices[num_train:num_train + num_val]] = True
    data.test_mask = torch.zeros(num_nodes, dtype=torch.bool); data.test_mask[indices[num_train + num_val:]] = True

    return data, len(np.unique(y_numpy))


In [6]:
# --- 5. 定义模型 ---
class DualConceptGCNTransformerWithCPE(nn.Module):
    def __init__(self,
                 raw_in_channels,
                 cpe_pos_channels,
                 cpe_neg_channels,
                 gcn_hidden_channels,
                 transformer_hidden_channels,
                 out_channels,
                 heads=1,
                 dropout=0.5):
        super(DualConceptGCNTransformerWithCPE, self).__init__()
        self.dropout = dropout

        # 正负分支先分别在对应概念图上做浅层 GCN 消息传递。
        self.pos_gcn = GCNConv(raw_in_channels, gcn_hidden_channels)
        self.neg_gcn = GCNConv(raw_in_channels, gcn_hidden_channels)

        # GCN 输出再与本分支的 CPE 拼接，作为 Graph Transformer 的输入。
        self.pos_transformer = TransformerConv(
            gcn_hidden_channels + cpe_pos_channels,
            transformer_hidden_channels,
            heads=heads,
            edge_dim=1
        )
        self.neg_transformer = TransformerConv(
            gcn_hidden_channels + cpe_neg_channels,
            transformer_hidden_channels,
            heads=heads,
            edge_dim=1
        )

        self.fusion_layer = nn.Linear(transformer_hidden_channels * heads * 2, out_channels)

    def forward(self,
                x_raw,
                cpe_pos,
                cpe_neg,
                edge_index_pos,
                edge_weight_pos,
                edge_attr_pos,
                edge_index_neg,
                edge_weight_neg,
                edge_attr_neg):
        h_pos_gcn = self.pos_gcn(x_raw, edge_index_pos, edge_weight=edge_weight_pos)
        h_pos_gcn = F.relu(h_pos_gcn)
        h_pos_gcn = F.dropout(h_pos_gcn, p=self.dropout, training=self.training)
        h_pos_input = torch.cat([h_pos_gcn, cpe_pos], dim=1)

        h_neg_gcn = self.neg_gcn(x_raw, edge_index_neg, edge_weight=edge_weight_neg)
        h_neg_gcn = F.relu(h_neg_gcn)
        h_neg_gcn = F.dropout(h_neg_gcn, p=self.dropout, training=self.training)
        h_neg_input = torch.cat([h_neg_gcn, cpe_neg], dim=1)

        h_pos = self.pos_transformer(h_pos_input, edge_index_pos, edge_attr_pos)
        h_pos = F.relu(h_pos)
        h_pos = F.dropout(h_pos, p=self.dropout, training=self.training)

        h_neg = self.neg_transformer(h_neg_input, edge_index_neg, edge_attr_neg)
        h_neg = F.relu(h_neg)
        h_neg = F.dropout(h_neg, p=self.dropout, training=self.training)

        h_combined = torch.cat([h_pos, h_neg], dim=1)
        out = self.fusion_layer(h_combined)
        return out


In [7]:
# --- 6. 实例化数据和模型 ---
data, num_classes = load_and_prepare_data(hparams['dataset'],
                                          hparams['threshold_pos'],
                                          hparams['threshold_neg'],
                                          hparams['cpe_profile_bins'])

model = DualConceptGCNTransformerWithCPE(
    raw_in_channels=data.x_raw.shape[1],
    cpe_pos_channels=data.cpe_pos.shape[1],
    cpe_neg_channels=data.cpe_neg.shape[1],
    gcn_hidden_channels=hparams['gcn_hidden_channels'],
    transformer_hidden_channels=hparams['transformer_hidden_channels'],
    out_channels=num_classes,
    heads=hparams['heads'],
    dropout=hparams['dropout']
)

print(f"GCN 输入维度: {data.x_raw.shape[1]}")
print(f"正分支 Transformer 输入维度: {hparams['gcn_hidden_channels'] + data.cpe_pos.shape[1]}")
print(f"负分支 Transformer 输入维度: {hparams['gcn_hidden_channels'] + data.cpe_neg.shape[1]}")

optimizer = torch.optim.Adam(model.parameters(), lr=hparams['learning_rate'], weight_decay=hparams['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()


原始特征维度: 43
正概念 CPE 维度: 9
负概念 CPE 维度: 9
正概念图边数: 1288
负概念图边数: 916
GCN 输入维度: 43
正分支 Transformer 输入维度: 25
负分支 Transformer 输入维度: 25


In [8]:
# --- 7. 训练与评估函数 ---
def run_model():
    return model(
        data.x_raw,
        data.cpe_pos,
        data.cpe_neg,
        data.edge_index_pos,
        data.edge_weight_pos,
        data.edge_attr_pos,
        data.edge_index_neg,
        data.edge_weight_neg,
        data.edge_attr_neg
    )


def train(epoch):
    model.train()
    optimizer.zero_grad()
    out = run_model()
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    writer.add_scalar('Loss/train', loss.item(), epoch)
    return loss.item()


def evaluate(epoch):
    model.eval()
    with torch.no_grad():
        out = run_model()
        pred = out.argmax(dim=1)

        train_acc = (pred[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()

        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/validation', val_acc, epoch)
        writer.add_scalar('Accuracy/test', test_acc, epoch)

        return train_acc, val_acc, test_acc


In [9]:
# --- 8. 主训练循环 ---
print("\n--- 开始训练 (GCN + profile8 CPE + 双概念格 Graph Transformer) ---")
for epoch in range(1, hparams['epochs'] + 1):
    loss = train(epoch)
    if epoch % 1 == 0:
        train_acc, val_acc, test_acc = evaluate(epoch)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

final_train_acc, final_val_acc, final_test_acc = evaluate(hparams['epochs'])
print('--- 训练完成 ---')
print(f'最终测试集准确率: {final_test_acc:.4f}')

metrics = {'accuracy/final_train': final_train_acc, 'accuracy/final_validation': final_val_acc, 'accuracy/final_test': final_test_acc}
writer.add_hparams(hparams, metrics)
writer.close()



--- 开始训练 (GCN + profile8 CPE + 双概念格 Graph Transformer) ---
Epoch: 001, Loss: 1.9928, Train Acc: 0.2167, Val Acc: 0.2500, Test Acc: 0.2381
Epoch: 002, Loss: 1.8319, Train Acc: 0.5500, Val Acc: 0.5500, Test Acc: 0.5238
Epoch: 003, Loss: 1.6917, Train Acc: 0.6000, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 004, Loss: 1.5677, Train Acc: 0.6000, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 005, Loss: 1.5385, Train Acc: 0.6000, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 006, Loss: 1.4655, Train Acc: 0.6000, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 007, Loss: 1.3696, Train Acc: 0.6000, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 008, Loss: 1.2770, Train Acc: 0.6167, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 009, Loss: 1.2455, Train Acc: 0.6167, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 010, Loss: 1.2078, Train Acc: 0.6167, Val Acc: 0.6000, Test Acc: 0.5238
Epoch: 011, Loss: 1.1209, Train Acc: 0.6167, Val Acc: 0.6000, Test Acc: 0.5714
Epoch: 012, Loss: 1.0332, Train Acc: 0.7000, Val Acc: 0.6000, Test Acc:

Epoch: 024, Loss: 0.4931, Train Acc: 0.9333, Val Acc: 0.9000, Test Acc: 0.8095
Epoch: 025, Loss: 0.4850, Train Acc: 0.9333, Val Acc: 0.9000, Test Acc: 0.8571
Epoch: 026, Loss: 0.4884, Train Acc: 0.9167, Val Acc: 0.8500, Test Acc: 0.8571
Epoch: 027, Loss: 0.4572, Train Acc: 0.9167, Val Acc: 0.8500, Test Acc: 0.8571
Epoch: 028, Loss: 0.4162, Train Acc: 0.9167, Val Acc: 0.9000, Test Acc: 0.8571


Epoch: 029, Loss: 0.3130, Train Acc: 0.9167, Val Acc: 0.9000, Test Acc: 0.8571
Epoch: 030, Loss: 0.3573, Train Acc: 0.9167, Val Acc: 0.9000, Test Acc: 0.8571
Epoch: 031, Loss: 0.3076, Train Acc: 0.9500, Val Acc: 0.9000, Test Acc: 0.8571
Epoch: 032, Loss: 0.2918, Train Acc: 0.9500, Val Acc: 0.9000, Test Acc: 0.8571
Epoch: 033, Loss: 0.3636, Train Acc: 0.9500, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 034, Loss: 0.2729, Train Acc: 0.9500, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 035, Loss: 0.1935, Train Acc: 0.9500, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 036, Loss: 0.2366, Train Acc: 0.9500, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 037, Loss: 0.1728, Train Acc: 0.9500, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 038, Loss: 0.3067, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571


Epoch: 039, Loss: 0.2893, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 040, Loss: 0.2881, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 041, Loss: 0.2037, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 042, Loss: 0.2259, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 043, Loss: 0.2214, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 044, Loss: 0.1519, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 045, Loss: 0.1555, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 046, Loss: 0.1538, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 047, Loss: 0.1075, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 048, Loss: 0.1188, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 049, Loss: 0.1560, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 050, Loss: 0.1275, Train Acc: 0.9667, Val Acc: 0.9500, Test Acc: 0.8571
Epoch: 051, Loss: 0.2312, Train Acc: 0.9667, Val Acc

Epoch: 074, Loss: 0.1117, Train Acc: 0.9833, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 075, Loss: 0.0555, Train Acc: 0.9833, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 076, Loss: 0.0551, Train Acc: 0.9833, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 077, Loss: 0.1087, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571


Epoch: 078, Loss: 0.0823, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 079, Loss: 0.0786, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 080, Loss: 0.0566, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 081, Loss: 0.0703, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 082, Loss: 0.0638, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 083, Loss: 0.1364, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 084, Loss: 0.0599, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 085, Loss: 0.0784, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 086, Loss: 0.1038, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 087, Loss: 0.0540, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 088, Loss: 0.1477, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 089, Loss: 0.1065, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 090, Loss: 0.0538, Train Acc: 1.0000, Val Acc

Epoch: 117, Loss: 0.0147, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 118, Loss: 0.1001, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 119, Loss: 0.0396, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 120, Loss: 0.0597, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 121, Loss: 0.0711, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 122, Loss: 0.0635, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 123, Loss: 0.0244, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 124, Loss: 0.0441, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 125, Loss: 0.0339, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 126, Loss: 0.0188, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571


Epoch: 127, Loss: 0.0533, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 128, Loss: 0.0396, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 129, Loss: 0.0515, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 130, Loss: 0.0305, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.8571
Epoch: 131, Loss: 0.0586, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 132, Loss: 0.0597, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 133, Loss: 0.0313, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 134, Loss: 0.0230, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 135, Loss: 0.0586, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 136, Loss: 0.0947, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 137, Loss: 0.0408, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 138, Loss: 0.0232, Train Acc: 1.0000, Val Acc: 1.0000, Test Acc: 0.9048
Epoch: 139, Loss: 0.0569, Train Acc: 1.0000, Val Acc